# AN-RA iterate500 T4 Training
Canonical bootstrap, preflight, 500M-class frontier training, resume, and ThirdEye evaluation. Secrets must be supplied through Colab secrets/environment variables, never notebook cells.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Wrong Colab runtime. Choose Runtime -> Change runtime type -> T4 GPU. TPU v5e/CPU should use the TPU notebook.')
print('gpu:', torch.cuda.get_device_name(0))
!nvidia-smi

In [ ]:
REPO = '/content/An-Ra-the-new-AGI'
BRANCH = 'iterate500'
![ -d "$REPO/.git" ] || git clone --branch "$BRANCH" --single-branch https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git "$REPO"
%cd $REPO
!git fetch origin "$BRANCH"
!git checkout "$BRANCH"
!git pull --ff-only origin "$BRANCH"
!python scripts/colab_bootstrap.py --repo "$REPO" --drive-root /content/drive/MyDrive/AnRa --install --install-thirdeye --model-size frontier

In [ ]:
%cd /content/An-Ra-the-new-AGI
from pathlib import Path
DATA_PROFILE = 't4-15gb'
FORCE_DATA_REBUILD = False
data_ready = Path('training_data/base_corpus.txt').exists() and Path('training_data/reasoning.jsonl').exists()
if FORCE_DATA_REBUILD or not data_ready:
    !python scripts/download_training_data.py --profile $DATA_PROFILE --prepare-corpus
else:
    print(f'Data already prepared for this runtime. Using profile={DATA_PROFILE}.')
!python -m data.causal_corpus
!python scripts/evaluate_with_thirdeye.py --profile quick --without-model

In [ ]:
import os
os.environ.setdefault('ANRA_THIRDEYE_INTELLIGENCE', '1')
SESSION_MINUTES = 180
!python scripts/build_brain.py --data_path training_data/anra_training.txt --checkpoint_path anra_frontier_500m.pt --model-size frontier --batch_size 1 --max_minutes $SESSION_MINUTES
!python scripts/evaluate_with_thirdeye.py --profile quick